## Module 1-2 Opening files "`with`" Python Context Manager

Everything we have done so far lived inside Python's memory: restart the kernel and all the variables are gone. Real research work needs to **read data from files** (a `.txt` file of a 10-K filing, a `.csv` file of Compustat data) and **write results back to files**.

A file is a *resource* we borrow from the operating system, and every borrowed resource has to be given back (**closed**) when we are done with it. A **context manager** — the `with` statement — is Python's way of saying:

> *"Borrow this resource for the length of this block, and give it back automatically — no matter what happens inside."*

Think of the fridge door: you open it, take the milk, and close it. `with` closes the door for you, even if you drop the milk on the way out.

### 1. Life without `with`: `open()` ... `close()`

Working with a file manually takes three steps: **open** it, **use** it, **close** it.

In [ ]:
# 1. open the file  ("w" = write mode, see the table in 6.3)
f = open("my_notes.txt", "w")

# 2. use it
f.write("Python for AccFin research\n")

# 3. close it  <- very easy to forget!
f.close()

print("Is the file closed?", f.closed)

Forgetting `close()` is not just untidy: what you wrote may never reach the disk, and the file stays locked by Python.

And it is easy to forget *by accident* — if an error happens between `open()` and `close()`, the `close()` line is simply never reached (remember Section 5).

In [ ]:
f = open("my_notes.txt", "w")

try:
    f.write("First line\n")
    print(10 + '50')      # <- If an Error happens here
    f.close()             # <- this line will never be reached
except TypeError as e:
    print("Error:", e)

print("Is the file closed?", f.closed)   # False -> the file is still open!

In [ ]:
# we have to clean up by hand
f.close()
print("Is the file closed now?", f.closed)

### 2. The `with` statement: the Pythonic way

```python
with open(file_name, mode) as f:
    # indented block = the "context"
    # the file object is available here under the name f
    ...
# outside the block: the file is already closed
```

- `open(file_name, mode)` gives us a **file object**,
- `as f` gives that object the name `f` (any name works; `f` and `file` are the common ones),
- everything **indented** below the `with` line is the context,
- when the block ends — normally **or** because of an error — Python calls `f.close()` for us.

so no `close()` needed.

In [ ]:
with open("my_notes.txt", "w") as f:
    f.write("Python for AccFin research\n")
    f.write("This line was written inside the with block\n")
# the block has ended here (notice the indentation stops)

print("Is the file closed?", f.closed)   # True -> closed automatically

In [ ]:
try:
    with open("my_notes.txt", "w") as f:
        f.write("First line\n")
        print(10 + '50')     # <- the same TypeError
except TypeError as e:
    print("Error:", e)

print("Is the file closed?", f.closed)   # True -> 'with' closed it anyway

### 3. File modes

The second argument of `open()` tells Python *what we intend to do* with the file:

| mode | meaning | if the file does not exist | if it does exist |
|---|---|---|---|
| `"r"` | **read** (the default) | `FileNotFoundError` | reads it |
| `"w"` | **write** | creates it | **erases everything** in it |
| `"a"` | **append** | creates it | adds to the end, keeps the old content |
| `"x"` | **create** | creates it | `FileExistsError` (a safe `"w"`) |

Two more things worth knowing:

- add a `"b"` (e.g. `"rb"`, `"wb"`) for **binary** files such as PDFs or Excel files — no `"b"` means plain text;
- always add `encoding="utf-8"` for text, otherwise Windows and Mac may disagree about accents and special characters: `open("filing.txt", "r", encoding="utf-8")`.

⚠️ `"w"` is the classic beginner trap: opening an existing file in `"w"` mode wipes it out immediately. Use `"a"` when you want to keep what is already there.

In [ ]:
# "a" = append: keep what is in the file and add one more line at the end
with open("my_notes.txt", "a", encoding="utf-8") as f:
    f.write("A line added later in append mode\n")

### 4. Reading a file back

There are three common ways to read a text file.

(Look carefully at the output below: the crashing cell in section 2 above re-opened the file in `"w"` mode and wiped it. That is the `"w"` trap in action.)

In [ ]:
# (1) .read()  ->  the WHOLE file as one single string
with open("my_notes.txt", "r", encoding="utf-8") as f:
    content = f.read()

print(type(content))
print(content)

In [ ]:
# (2) .readlines()  ->  a LIST of strings, one per line (each still ends with "\n")
with open("my_notes.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(lines)
print("Number of lines:", len(lines))

In [ ]:
# (3) loop over the file line by line: reads one line at a time,
#     so it works even for a 200 MB file that would not fit in memory
with open("my_notes.txt", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        print(i, line.strip())     # .strip() removes the trailing "\n"

### 5. A research example: reading a keyword list

The workshop's shared datasets live in the `data/` folder. `data/R&D_Keywords.txt` is a plain text file with one R&D-related phrase per line — the kind of keyword list used to measure innovation disclosure in a 10-K.

File paths are relative to the **working directory**, i.e. the folder Python thinks it is sitting in. That is why the same path works for your classmate and fails for you: you launched Jupyter from a different folder. `os.getcwd()` tells you where you are.

In [ ]:
path = "../data/R&D_Keywords.txt"

with open(path, "r", encoding="utf-8") as f:
    keywords = [line.strip() for line in f]   # list comprehension, see Section 2.3

print("Number of keywords:", len(keywords))
print(keywords[:5])

### 6. Two files in one `with`

A very common research task is *read one file, write another*. You can open both in a single `with` line, separated by a comma — both get closed at the end of the block.

In [ ]:
with open(path, "r", encoding="utf-8") as infile, open("keywords_upper.txt", "w", encoding="utf-8") as outfile:
    for line in infile:
        outfile.write(line.upper())

# check the result
with open("keywords_upper.txt", "r", encoding="utf-8") as f:
    print(f.readlines()[:3])

### 7. `with` is not only about files

Once you recognise the pattern, you will see it everywhere in the rest of the workshop — anything that has to be *opened and closed*, *started and stopped*, or *saved and released*:

```python
with wrds.Connection(wrds_username="...") as db:   # closes the WRDS connection
    data = db.raw_sql("select * from comp.funda limit 10")

with pd.ExcelWriter("results.xlsx") as writer:     # writes and saves the Excel file
    df.to_excel(writer, sheet_name="Table 1")

with requests.Session() as s:                      # releases the network connection
    r = s.get("https://www.sec.gov/...")
```

**The rule of thumb: if a library gives you something you are supposed to `close()`, open it with `with`.**